# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q factor_analyzer pandas numpy matplotlib seaborn
    print('Packages installed.')


# Exploratory Factor Analysis (EFA) In-Depth
This notebook demonstrates how to perform Exploratory Factor Analysis (EFA) to uncover latent structures in survey data. We use the `factor_analyzer` library to identify underlying factors (e.g., personality traits) from a set of observed variables (Likert items).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

sns.set_style("whitegrid")

## 1. Generate Synthetic Survey Data
We simulate data for 500 respondents on 10 Likert-scale items (1-7).
We intentionally design the data to have 2 underlying factors:
*   **Factor 1 (e.g., Anxiety)**: Items Q1, Q2, Q3, Q4, Q5
*   **Factor 2 (e.g., Extraversion)**: Items Q6, Q7, Q8, Q9, Q10

In [ ]:
# Sample size
N = 500

# Two latent variables (uncorrelated for simplicity)
latent_f1 = np.random.normal(0, 1, N)
latent_f2 = np.random.normal(0, 1, N)

# Generate 10 items
data = pd.DataFrame()

# Items 1-5 load on Factor 1
for i in range(1, 6):
    # Item = Loading * Factor + Error
    item = 0.8 * latent_f1 + np.random.normal(0, 0.6, N)
    data[f'Q{i}'] = item

# Items 6-10 load on Factor 2
for i in range(6, 11):
    item = 0.7 * latent_f2 + np.random.normal(0, 0.7, N)
    data[f'Q{i}'] = item

# Scale to Likert (1-7) and round
data = (data - data.min()) / (data.max() - data.min()) * 6 + 1
data = data.round().astype(int)

print("Sample Data:")
print(data.head())

## 2. Assumptions Testing
Before running EFA, we check if the data is suitable.
*   **Bartlett's Test of Sphericity**: Checks if observed variables correlate at all. (p < 0.05 is good)
*   **Kaiser-Meyer-Olkin (KMO)**: Measure of sampling adequacy. (> 0.6 is good)

In [ ]:
chi_square_value, p_value = calculate_bartlett_sphericity(data)
print(f"Bartlett's Test: Chi^2={chi_square_value:.2f}, p-value={p_value:.4f}")

kmo_all, kmo_model = calculate_kmo(data)
print(f"KMO Test: {kmo_model:.3f}")

## 3. Determining Number of Factors (Scree Plot)
We look for the "elbow" in the eigenvalues.

In [ ]:
fa = FactorAnalyzer(n_factors=10, rotation=None)
fa.fit(data)
ev, v = fa.get_eigenvalues()

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), ev, marker='o')
plt.title('Scree Plot')
plt.xlabel('Factors')
plt.ylabel('Eigenvalue')
plt.axhline(1, color='r', linestyle='--') # Kaiser criterion
plt.grid()
plt.show()

## 4. Factor Analysis with Rotation
Based on the scree plot (likely 2 factors), we run the final model using Varimax rotation (orthogonal) or Promax (oblique). We'll use Varimax here.

In [ ]:
fa = FactorAnalyzer(n_factors=2, rotation='varimax')
fa.fit(data)

# Factor Loadings
loadings = pd.DataFrame(fa.loadings_, index=data.columns, columns=['Factor 1', 'Factor 2'])
print(loadings)

# Visualize
plt.figure(figsize=(6, 8))
sns.heatmap(loadings, annot=True, cmap='coolwarm', center=0)
plt.title('Factor Loadings')
plt.show()

## 5. Factor Variance
How much variance does each factor explain?

In [ ]:
variance = pd.DataFrame(fa.get_factor_variance(), 
                        index=['SS Loadings', 'Proportion Var', 'Cumulative Var'],
                        columns=['Factor 1', 'Factor 2'])
print(variance)